In [1]:
import pandas as pd
import requests
from datetime import datetime, timedelta
import os
import logging
from concurrent.futures import ThreadPoolExecutor, as_completed
import time
from urllib.parse import urlparse
import numpy as np
from io import StringIO
import json
import re

# Logging konfigurieren
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

class BundesbankDataAnalyzer:
    def __init__(self, data_dir="downloaded_data", enhanced_output_dir="enhanced_data"):
        self.data_dir = data_dir
        self.enhanced_output_dir = enhanced_output_dir
        
        # Verzeichnisse erstellen
        for directory in [data_dir, enhanced_output_dir]:
            if not os.path.exists(directory):
                os.makedirs(directory)
                
        self.session = requests.Session()
        self.session.headers.update({
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
        })
    
    def parse_bundesbank_csv(self, csv_content, indicator_code):
        """
        Parst spezifisch formatierte Bundesbank CSV-Dateien und extrahiert Metadaten
        """
        try:
            lines = csv_content.strip().split('\n')
            
            # Initialisiere Analyse-Objekt
            analysis = {
                'parsing_method': 'bundesbank_format',
                'raw_lines_count': len(lines)
            }
            
            # Extrahiere Metadaten aus den ersten Zeilen
            metadata = {}
            time_series_data = []
            data_start_line = None
            
            for i, line in enumerate(lines):
                line = line.strip()
                if not line:
                    continue
                
                # Parse Metadaten-Zeilen
                if ';' in line:
                    parts = [part.strip(' "') for part in line.split(';')]
                    
                    # Spezielle Metadaten-Zeilen erkennen
                    if len(parts) >= 2:
                        key = parts[0].lower()
                        value = parts[1] if len(parts) > 1 else ''
                        
                        # Bekannte Metadaten-Felder
                        if key == 'dezimalstellen':
                            metadata['dezimalstellen'] = value
                        elif key == 'dimension':
                            metadata['dimension'] = value
                        elif key == 'einheit':
                            metadata['einheit'] = value
                        elif key == 'format der zeitangabe':
                            metadata['format_zeitangabe'] = value
                        elif key == 'kategorie':
                            metadata['kategorie'] = value
                        elif key == 'stand vom':
                            metadata['stand_vom'] = value
                        elif self.is_time_period(parts[0]):
                            # Zeitreihen-Daten gefunden
                            if data_start_line is None:
                                data_start_line = i
                            
                            period = parts[0]
                            value_str = parts[1] if len(parts) > 1 else ''
                            
                            # Versuche Wert zu konvertieren (Komma als Dezimaltrennzeichen)
                            try:
                                if value_str and value_str not in ['', '.', '..', 'N/A']:
                                    value_clean = value_str.replace(',', '.')
                                    value_numeric = float(value_clean)
                                    time_series_data.append({
                                        'period': period,
                                        'value': value_numeric,
                                        'original_value': value_str
                                    })
                                else:
                                    time_series_data.append({
                                        'period': period,
                                        'value': None,
                                        'original_value': value_str
                                    })
                            except ValueError:
                                time_series_data.append({
                                    'period': period,
                                    'value': None,
                                    'original_value': value_str
                                })
            
            # Übertrage extrahierte Metadaten
            analysis.update(metadata)
            
            # Analysiere Zeitreihen-Daten
            if time_series_data:
                analysis['total_periods'] = len(time_series_data)
                analysis['data_start_line'] = data_start_line
                
                # Extrahiere Min/Max Datum
                periods = [item['period'] for item in time_series_data if item['period']]
                if periods:
                    analysis['min_datum'] = min(periods)
                    analysis['max_datum'] = max(periods)
                
                # Analysiere Werte
                values = [item['value'] for item in time_series_data if item['value'] is not None]
                if values:
                    analysis['total_valid_values'] = len(values)
                    analysis['min_value'] = min(values)
                    analysis['max_value'] = max(values)
                    analysis['mean_value'] = sum(values) / len(values)
                    
                    # Letzte verfügbare Werte
                    valid_data = [item for item in time_series_data if item['value'] is not None]
                    if valid_data:
                        analysis['latest_period'] = valid_data[-1]['period']
                        analysis['latest_value'] = valid_data[-1]['value']
                        analysis['latest_value_original'] = valid_data[-1]['original_value']
                        
                        if len(valid_data) >= 2:
                            analysis['second_latest_value'] = valid_data[-2]['value']
                            analysis['latest_change'] = valid_data[-1]['value'] - valid_data[-2]['value']
                            if valid_data[-2]['value'] != 0:
                                analysis['latest_change_percent'] = (analysis['latest_change'] / valid_data[-2]['value']) * 100
                
                # Schätze Frequenz basierend auf Period-Format
                analysis['estimated_frequency'] = self.estimate_frequency_from_periods(periods)
                
                # Datenqualität
                analysis['data_completeness_percent'] = (len(values) / len(time_series_data)) * 100 if time_series_data else 0
                
                # Konvertiere Stand vom zu datetime falls möglich
                if 'stand_vom' in analysis:
                    try:
                        # Format: "30.01.2024 10:08:22 Uhr"
                        stand_clean = analysis['stand_vom'].replace(' Uhr', '')
                        stand_dt = datetime.strptime(stand_clean, '%d.%m.%Y %H:%M:%S')
                        analysis['stand_vom_datetime'] = stand_dt.isoformat()
                    except:
                        pass
                
                # Erstelle DataFrame für weitere Analyse
                analysis['time_series_df'] = pd.DataFrame(time_series_data)
            
            analysis['parsing_status'] = 'success'
            return analysis
            
        except Exception as e:
            logging.error(f"Fehler beim Parsen der Bundesbank CSV für {indicator_code}: {e}")
            return {
                'parsing_status': 'failed',
                'parsing_error': str(e),
                'parsing_method': 'bundesbank_format'
            }
    
    def is_time_period(self, text):
        """
        Prüft ob ein Text ein Zeitraum ist (z.B. 1991-Q1, 2024-03, 2024)
        """
        if not text or not isinstance(text, str):
            return False
        
        # Bekannte Zeitreihen-Formate
        patterns = [
            r'^\d{4}-Q[1-4]$',      # 1991-Q1
            r'^\d{4}-\d{2}$',       # 2024-03
            r'^\d{4}$',             # 2024
            r'^\d{4}-W\d{2}$',      # 2024-W15
            r'^\d{4}-\d{2}-\d{2}$'  # 2024-03-15
        ]
        
        for pattern in patterns:
            if re.match(pattern, text.strip()):
                return True
        
        return False
    
    def estimate_frequency_from_periods(self, periods):
        """
        Schätzt die Frequenz basierend auf den Period-Strings
        """
        if not periods:
            return 'Unknown'
        
        sample_period = periods[0]
        
        if '-Q' in sample_period:
            return 'Quarterly'
        elif re.match(r'\d{4}-\d{2}$', sample_period):
            return 'Monthly'
        elif re.match(r'\d{4}$', sample_period):
            return 'Annual'
        elif '-W' in sample_period:
            return 'Weekly'
        elif re.match(r'\d{4}-\d{2}-\d{2}$', sample_period):
            return 'Daily'
        else:
            return 'Irregular'
    
    def download_csv_data(self, csv_url, indicator_code, max_retries=3):
        """
        Lädt CSV-Daten für einen Indikator herunter und analysiert sie mit Bundesbank-Parser
        """
        for attempt in range(max_retries):
            try:
                response = self.session.get(csv_url, timeout=30)
                response.raise_for_status()
                time.sleep(1)  # Kurze Pause zwischen den Versuchen  
                
                # CSV-Daten dekodieren (UTF-8 mit BOM-Behandlung)
                csv_content = response.content.decode('utf-8-sig')
                
                # Datei lokal speichern
                filename = f"{indicator_code.replace('.', '_')}.csv"
                filepath = os.path.join(self.data_dir, filename)
                
                with open(filepath, 'w', encoding='utf-8', newline='') as f:
                    f.write(csv_content)
                
                # Bundesbank-spezifisches Parsing
                analysis = self.parse_bundesbank_csv(csv_content, indicator_code)
                
                # Zusätzliche Download-Informationen
                analysis['download_status'] = 'success'
                analysis['local_file'] = filepath
                analysis['download_timestamp'] = datetime.now().isoformat()
                analysis['file_size_bytes'] = len(csv_content.encode('utf-8'))
                
                return analysis
                
            except requests.exceptions.RequestException as e:
                if attempt < max_retries - 1:
                    logging.warning(f"Download-Versuch {attempt + 1} für {indicator_code} fehlgeschlagen: {e}")
                    time.sleep(2 ** attempt)
                else:
                    logging.error(f"Download für {indicator_code} nach {max_retries} Versuchen fehlgeschlagen: {e}")
                    return {
                        'download_status': 'failed',
                        'error_message': str(e),
                        'download_timestamp': datetime.now().isoformat()
                    }
            except Exception as e:
                logging.error(f"Unerwarteter Fehler beim Download von {indicator_code}: {e}")
                return {
                    'download_status': 'failed',
                    'error_message': f"Unerwarteter Fehler: {str(e)}",
                    'download_timestamp': datetime.now().isoformat()
                }
    
    def enhance_scraping_results(self, scraping_file_path, output_file=None, max_workers=5, limit_downloads=None):
        """
        Erweitert die Scraping-Ergebnisse um Download- und Analysedaten mit Bundesbank-spezifischem Parsing
        """
        try:
            # Scraping-Ergebnisse laden
            df_scraping = pd.read_excel(scraping_file_path)
            logging.info(f"Scraping-Datei geladen: {len(df_scraping)} Einträge")
            
            # Nur erfolgreiche Einträge mit CSV-Links verarbeiten
            successful_entries = df_scraping[
                (df_scraping['status'] == 'success') & 
                (df_scraping['csv_link'].notna()) & 
                (~df_scraping['csv_link'].str.contains('nicht gefunden', na=False))
            ].copy()
            
            if limit_downloads:
                successful_entries = successful_entries.head(limit_downloads)
                logging.info(f"Limitiere Downloads auf {limit_downloads} Einträge")
            
            logging.info(f"Verarbeite {len(successful_entries)} erfolgreiche Einträge")
            
            # Parallel Downloads und Analyse
            enhanced_data = []
            
            def download_and_analyze(row):
                indicator_code = row['indicator_code']
                csv_link = row['csv_link']
                
                logging.info(f"Verarbeite: {indicator_code}")
                analysis = self.download_csv_data(csv_link, indicator_code)
                
                # Ursprüngliche Daten mit Analysedaten kombinieren
                enhanced_row = row.to_dict()
                enhanced_row.update(analysis)
                
                return enhanced_row
            
            # Threading für Downloads
            with ThreadPoolExecutor(max_workers=max_workers) as executor:
                futures = {executor.submit(download_and_analyze, row): idx 
                          for idx, row in successful_entries.iterrows()}
                
                completed = 0
                for future in as_completed(futures):
                    try:
                        enhanced_row = future.result()
                        enhanced_data.append(enhanced_row)
                        completed += 1
                        
                        if completed % 10 == 0:
                            logging.info(f"Fortschritt: {completed}/{len(successful_entries)} abgeschlossen")
                            
                    except Exception as e:
                        logging.error(f"Fehler bei Download/Analyse: {e}")
            
            # Erweiterte DataFrame erstellen
            df_enhanced = pd.DataFrame(enhanced_data)
            
            # Nicht heruntergeladene Einträge hinzufügen
            failed_entries = df_scraping[
                ~df_scraping.index.isin(successful_entries.index)
            ].copy()
            
            for col in df_enhanced.columns:
                if col not in failed_entries.columns:
                    failed_entries[col] = None
            
            # Kombinieren
            df_final = pd.concat([df_enhanced, failed_entries], ignore_index=True)
            
            # Ausgabedatei speichern
            if output_file is None:
                timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
                output_file = os.path.join(self.enhanced_output_dir, f"enhanced_bundesbank_data_{timestamp}.xlsx")
            
            # Spalten sortieren für bessere Übersicht - Bundesbank-spezifische Felder zuerst
            priority_columns = [
                'indicator_code', 'title', 'status', 'download_status', 'parsing_status',
                # Bundesbank Metadaten
                'dezimalstellen', 'dimension', 'einheit', 'format_zeitangabe', 'kategorie', 'stand_vom',
                # Zeitreihen-Info  
                'min_datum', 'max_datum', 'total_periods', 'estimated_frequency',
                'latest_period', 'latest_value', 'latest_change_percent',
                # Datenqualität
                'data_completeness_percent', 'total_valid_values',
                # Statistiken
                'min_value', 'max_value', 'mean_value',
                # Links und Dateien
                'csv_link', 'xml1_link', 'xml2_link', 'local_file'
            ]
            
            remaining_columns = [col for col in df_final.columns if col not in priority_columns]
            ordered_columns = priority_columns + remaining_columns
            ordered_columns = [col for col in ordered_columns if col in df_final.columns]
            
            df_final = df_final[ordered_columns]
            
            # Mit mehreren Sheets speichern
            with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
                # Hauptdaten
                df_final.to_excel(writer, sheet_name='Enhanced_Data', index=False)
                
                # Zusammenfassung
                summary_data = self.create_summary(df_final)
                summary_df = pd.DataFrame(summary_data.items(), columns=['Metric', 'Value'])
                summary_df.to_excel(writer, sheet_name='Summary', index=False)
                
                # Erfolgreiche Downloads mit Bundesbank-Metadaten
                successful_downloads = df_final[
                    (df_final['download_status'] == 'success') & 
                    (df_final['parsing_status'] == 'success')
                ]
                if len(successful_downloads) > 0:
                    successful_downloads.to_excel(writer, sheet_name='Successful_Downloads', index=False)
                
                # Metadaten-Übersicht
                metadata_overview = self.create_metadata_overview(df_final)
                if metadata_overview is not None:
                    metadata_overview.to_excel(writer, sheet_name='Metadata_Overview', index=False)
                
                # Fehlgeschlagene Downloads
                failed_downloads = df_final[
                    (df_final['download_status'] == 'failed') | 
                    (df_final['parsing_status'] == 'failed')
                ]
                if len(failed_downloads) > 0:
                    failed_downloads.to_excel(writer, sheet_name='Failed_Downloads', index=False)
            
            logging.info(f"Erweiterte Daten gespeichert: {output_file}")
            
            # Detaillierte Statistiken
            successful_parsing = len(df_final[df_final['parsing_status'] == 'success'])
            successful_downloads = len(df_final[df_final['download_status'] == 'success'])
            
            logging.info(f"Erfolgreiche Downloads: {successful_downloads}")
            logging.info(f"Erfolgreiches Parsing: {successful_parsing}")
            
            # Bundesbank-spezifische Statistiken
            if successful_parsing > 0:
                with_metadata = len(df_final[df_final['dimension'].notna()])
                with_timedata = len(df_final[df_final['min_datum'].notna()])
                
                logging.info(f"Mit Metadaten: {with_metadata}")
                logging.info(f"Mit Zeitreihen-Daten: {with_timedata}")
            
            return df_final, output_file
            
        except Exception as e:
            logging.error(f"Fehler beim Erweitern der Scraping-Ergebnisse: {e}")
            return None, None
    
    def create_metadata_overview(self, df_enhanced):
        """
        Erstellt eine Übersicht über die Bundesbank-Metadaten
        """
        try:
            successful_data = df_enhanced[df_enhanced['parsing_status'] == 'success']
            
            if len(successful_data) == 0:
                return None
            
            overview_data = []
            
            # Dimensionen-Verteilung
            if 'dimension' in successful_data.columns:
                dimension_counts = successful_data['dimension'].value_counts()
                for dim, count in dimension_counts.items():
                    overview_data.append({
                        'Kategorie': 'Dimension',
                        'Wert': dim,
                        'Anzahl': count,
                        'Prozent': f"{count/len(successful_data)*100:.1f}%"
                    })
            
            # Einheiten-Verteilung
            if 'einheit' in successful_data.columns:
                einheit_counts = successful_data['einheit'].value_counts()
                for einheit, count in einheit_counts.items():
                    overview_data.append({
                        'Kategorie': 'Einheit',
                        'Wert': einheit,
                        'Anzahl': count,
                        'Prozent': f"{count/len(successful_data)*100:.1f}%"
                    })
            
            # Frequenz-Verteilung
            if 'estimated_frequency' in successful_data.columns:
                freq_counts = successful_data['estimated_frequency'].value_counts()
                for freq, count in freq_counts.items():
                    overview_data.append({
                        'Kategorie': 'Frequenz',
                        'Wert': freq,
                        'Anzahl': count,
                        'Prozent': f"{count/len(successful_data)*100:.1f}%"
                    })
            
            # Kategorien-Verteilung
            if 'kategorie' in successful_data.columns:
                kategorie_counts = successful_data['kategorie'].value_counts()
                for kat, count in kategorie_counts.items():
                    overview_data.append({
                        'Kategorie': 'BBK-Kategorie',
                        'Wert': kat,
                        'Anzahl': count,
                        'Prozent': f"{count/len(successful_data)*100:.1f}%"
                    })
            
            return pd.DataFrame(overview_data) if overview_data else None
            
        except Exception as e:
            logging.error(f"Fehler beim Erstellen der Metadaten-Übersicht: {e}")
            return None
    
    def create_summary(self, df_enhanced):
        """
        Erstellt eine Zusammenfassung der erweiterten Daten mit Bundesbank-spezifischen Metriken
        """
        summary = {
            'Total_Indicators': len(df_enhanced),
            'Successful_Scraping': len(df_enhanced[df_enhanced['status'] == 'success']),
            'Failed_Scraping': len(df_enhanced[df_enhanced['status'] == 'error']),
            'Successful_Downloads': len(df_enhanced[df_enhanced['download_status'] == 'success']),
            'Failed_Downloads': len(df_enhanced[df_enhanced['download_status'] == 'failed']),
            'Successful_Parsing': len(df_enhanced[df_enhanced['parsing_status'] == 'success']),
        }
        
        # Bundesbank-spezifische Metriken
        successful_data = df_enhanced[df_enhanced['parsing_status'] == 'success']
        
        if len(successful_data) > 0:
            # Metadaten-Verfügbarkeit
            summary['With_Metadata'] = len(successful_data[successful_data['dimension'].notna()])
            summary['With_Time_Data'] = len(successful_data[successful_data['min_datum'].notna()])
            
            # Häufigste Dimensionen/Einheiten
            if 'dimension' in successful_data.columns:
                top_dimension = successful_data['dimension'].mode()
                summary['Most_Common_Dimension'] = top_dimension.iloc[0] if len(top_dimension) > 0 else 'N/A'
            
            if 'einheit' in successful_data.columns:
                top_einheit = successful_data['einheit'].mode()
                summary['Most_Common_Unit'] = top_einheit.iloc[0] if len(top_einheit) > 0 else 'N/A'
            
            # Frequenz-Verteilung
            if 'estimated_frequency' in successful_data.columns:
                freq_counts = successful_data['estimated_frequency'].value_counts()
                for freq, count in freq_counts.items():
                    summary[f'Frequency_{freq}'] = count
            
            # Durchschnittliche Datenqualität
            if 'data_completeness_percent' in successful_data.columns:
                completeness = successful_data['data_completeness_percent'].dropna()
                if len(completeness) > 0:
                    summary['Average_Data_Completeness'] = f"{completeness.mean():.2f}%"
            
            # Zeitreihen-Abdeckung
            if 'min_datum' in successful_data.columns:
                valid_min_dates = successful_data['min_datum'].dropna()
                valid_max_dates = successful_data['max_datum'].dropna()
                
                if len(valid_min_dates) > 0:
                    summary['Earliest_Data_Point'] = min(valid_min_dates)
                    summary['Latest_Data_Point'] = max(valid_max_dates) if len(valid_max_dates) > 0 else 'N/A'
            
            # Durchschnittliche Zeitreihen-Länge
            if 'total_periods' in successful_data.columns:
                periods = successful_data['total_periods'].dropna()
                if len(periods) > 0:
                    summary['Average_Time_Series_Length'] = f"{periods.mean():.1f} periods"
        
        return summary
    
    def generate_data_report(self, enhanced_file_path):
        """
        Generiert einen detaillierten Bericht über die Bundesbank-Daten
        """
        try:
            df = pd.read_excel(enhanced_file_path, sheet_name='Enhanced_Data')
            
            report = {
                'generation_timestamp': datetime.now().isoformat(),
                'total_indicators': len(df),
                'bundesbank_analysis': {
                    'metadata_extraction': {},
                    'time_series_analysis': {},
                    'data_quality': {},
                    'coverage_analysis': {}
                },
                'recommendations': []
            }
            
            successful_data = df[df['parsing_status'] == 'success']
            
            if len(successful_data) > 0:
                # Metadaten-Extraktion Analyse
                report['bundesbank_analysis']['metadata_extraction'] = {
                    'total_with_metadata': len(successful_data[successful_data['dimension'].notna()]),
                    'metadata_extraction_rate': f"{len(successful_data[successful_data['dimension'].notna()])/len(successful_data)*100:.1f}%",
                    'unique_dimensions': successful_data['dimension'].nunique() if 'dimension' in successful_data.columns else 0,
                    'unique_units': successful_data['einheit'].nunique() if 'einheit' in successful_data.columns else 0,
                    'unique_categories': successful_data['kategorie'].nunique() if 'kategorie' in successful_data.columns else 0
                }
                
                # Zeitreihen-Analyse
                with_time_data = successful_data[successful_data['min_datum'].notna()]
                if len(with_time_data) > 0:
                    report['bundesbank_analysis']['time_series_analysis'] = {
                        'series_with_time_data': len(with_time_data),
                        'time_coverage_rate': f"{len(with_time_data)/len(successful_data)*100:.1f}%",
                        'earliest_data': with_time_data['min_datum'].min() if 'min_datum' in with_time_data.columns else 'N/A',
                        'latest_data': with_time_data['max_datum'].max() if 'max_datum' in with_time_data.columns else 'N/A',
                        'average_series_length': f"{with_time_data['total_periods'].mean():.1f}" if 'total_periods' in with_time_data.columns else 'N/A',
                        'frequency_distribution': with_time_data['estimated_frequency'].value_counts().to_dict() if 'estimated_frequency' in with_time_data.columns else {}
                    }
                
                # Datenqualität
                if 'data_completeness_percent' in successful_data.columns:
                    completeness = successful_data['data_completeness_percent'].dropna()
                    if len(completeness) > 0:
                        report['bundesbank_analysis']['data_quality'] = {
                            'average_completeness': f"{completeness.mean():.2f}%",
                            'high_quality_series': len(completeness[completeness >= 95]),
                            'medium_quality_series': len(completeness[(completeness >= 80) & (completeness < 95)]),
                            'low_quality_series': len(completeness[completeness < 80]),
                            'completeness_distribution': {
                                '95-100%': len(completeness[completeness >= 95]),
                                '80-95%': len(completeness[(completeness >= 80) & (completeness < 95)]),
                                '50-80%': len(completeness[(completeness >= 50) & (completeness < 80)]),
                                '<50%': len(completeness[completeness < 50])
                            }
                        }
                
                # Empfehlungen basierend auf Bundesbank-Daten
                if report['bundesbank_analysis']['metadata_extraction']['metadata_extraction_rate'] != '100.0%':
                    missing_metadata = len(successful_data) - len(successful_data[successful_data['dimension'].notna()])
                    report['recommendations'].append(f"Review {missing_metadata} series with missing metadata")
                
                if 'data_quality' in report['bundesbank_analysis']:
                    low_quality = report['bundesbank_analysis']['data_quality']['low_quality_series']
                    if low_quality > 0:
                        report['recommendations'].append(f"Investigate {low_quality} series with <80% data completeness")
                
                quarterly_series = report['bundesbank_analysis']['time_series_analysis'].get('frequency_distribution', {}).get('Quarterly', 0)
                monthly_series = report['bundesbank_analysis']['time_series_analysis'].get('frequency_distribution', {}).get('Monthly', 0)
                
                if quarterly_series > monthly_series:
                    report['recommendations'].append("Dataset is predominantly quarterly - consider quarterly analysis approach")
                elif monthly_series > quarterly_series:
                    report['recommendations'].append("Dataset is predominantly monthly - high-frequency analysis possible")
            
            # Report als JSON speichern
            report_file = enhanced_file_path.replace('.xlsx', '_report.json')
            with open(report_file, 'w', encoding='utf-8') as f:
                json.dump(report, f, indent=2, ensure_ascii=False)
            
            logging.info(f"Bundesbank-Datenreport erstellt: {report_file}")
            return report, report_file
            
        except Exception as e:
            logging.error(f"Fehler beim Erstellen des Bundesbank-Reports: {e}")
            return None, None


def main():
    analyzer = BundesbankDataAnalyzer()
    
    print("=== BUNDESBANK DATENANALYSE TOOL (Erweitert) ===")
    print("Extrahiert automatisch:")
    print("- Dezimalstellen, Dimension, Einheit") 
    print("- Format der Zeitangabe, Kategorie, Stand vom")
    print("- Min-Datum, Max-Datum (automatisch ermittelt)")
    print("- Vollständige Zeitreihen-Analyse")
    print()
    print("1. Erweitere Scraping-Ergebnisse (Download + Bundesbank-Analyse)")
    print("2. Generiere Bundesbank-Datenreport")
    print("3. Teste einzelne CSV-Datei")
    
    choice = input("Wählen Sie eine Option (1/2/3): ").strip()
    
    if choice == "1":
        scraping_file = input("Pfad zur Scraping-Excel-Datei: ").strip()
        if not os.path.exists(scraping_file):
            print("Datei nicht gefunden!")
            return
        
        limit = input("Download-Limit (Enter für alle): ").strip()
        limit_downloads = int(limit) if limit.isdigit() else None
        
        workers = input("Anzahl parallele Downloads (Standard: 5): ").strip()
        max_workers = int(workers) if workers.isdigit() else 5
        
        print(f"\nStarte Bundesbank-Analyse...")
        print("Extrahiert: Dezimalstellen, Dimension, Einheit, Zeitangaben, etc.")
        
        df_enhanced, output_file = analyzer.enhance_scraping_results(
            scraping_file, 
            max_workers=max_workers,
            limit_downloads=limit_downloads
        )
        
        if df_enhanced is not None:
            print(f"\n=== BUNDESBANK-ANALYSE ABGESCHLOSSEN ===")
            print(f"Ausgabedatei: {output_file}")
            
            # Zeige Bundesbank-spezifische Statistiken
            successful_parsing = len(df_enhanced[df_enhanced['parsing_status'] == 'success'])
            with_metadata = len(df_enhanced[df_enhanced['dimension'].notna()])
            with_timedata = len(df_enhanced[df_enhanced['min_datum'].notna()])
            
            print(f"\nBundesbank-Metadaten extrahiert:")
            print(f"- Erfolgreiches Parsing: {successful_parsing}")
            print(f"- Mit Dimension/Einheit: {with_metadata}")
            print(f"- Mit Zeitreihen-Daten: {with_timedata}")
            
            # Zeige Beispiel-Metadaten
            if successful_parsing > 0:
                sample = df_enhanced[df_enhanced['parsing_status'] == 'success'].iloc[0]
                print(f"\nBeispiel-Metadaten:")
                if pd.notna(sample.get('dimension')):
                    print(f"- Dimension: {sample.get('dimension')}")
                if pd.notna(sample.get('einheit')):
                    print(f"- Einheit: {sample.get('einheit')}")
                if pd.notna(sample.get('dezimalstellen')):
                    print(f"- Dezimalstellen: {sample.get('dezimalstellen')}")
                if pd.notna(sample.get('min_datum')):
                    print(f"- Zeitspanne: {sample.get('min_datum')} bis {sample.get('max_datum')}")
                if pd.notna(sample.get('estimated_frequency')):
                    print(f"- Frequenz: {sample.get('estimated_frequency')}")
            
            # Automatisch Report generieren
            report, report_file = analyzer.generate_data_report(output_file)
            if report:
                print(f"\nBundesbank-Report erstellt: {report_file}")
                
                # Zeige wichtige Report-Metriken
                if 'bundesbank_analysis' in report:
                    meta = report['bundesbank_analysis']['metadata_extraction']
                    print(f"- Metadaten-Extraktionsrate: {meta.get('metadata_extraction_rate', 'N/A')}")
                    print(f"- Unique Dimensionen: {meta.get('unique_dimensions', 'N/A')}")
                    print(f"- Unique Einheiten: {meta.get('unique_units', 'N/A')}")
    
    elif choice == "2":
        enhanced_file = input("Pfad zur erweiterten Excel-Datei: ").strip()
        if not os.path.exists(enhanced_file):
            print("Datei nicht gefunden!")
            return
        
        report, report_file = analyzer.generate_data_report(enhanced_file)
        if report:
            print(f"Bundesbank-Report erstellt: {report_file}")
            print("\nZusammenfassung:")
            
            # Zeige Bundesbank-spezifische Metriken
            if 'bundesbank_analysis' in report:
                ba = report['bundesbank_analysis']
                
                print("Metadaten-Extraktion:")
                for key, value in ba['metadata_extraction'].items():
                    print(f"  - {key}: {value}")
                
                if 'time_series_analysis' in ba:
                    print("Zeitreihen-Analyse:")
                    for key, value in ba['time_series_analysis'].items():
                        if key != 'frequency_distribution':
                            print(f"  - {key}: {value}")
                
                if 'data_quality' in ba:
                    print("Datenqualität:")
                    dq = ba['data_quality']
                    print(f"  - Durchschnittliche Vollständigkeit: {dq.get('average_completeness', 'N/A')}")
                    print(f"  - Hohe Qualität (≥95%): {dq.get('high_quality_series', 'N/A')}")
                    print(f"  - Niedrige Qualität (<80%): {dq.get('low_quality_series', 'N/A')}")
    
    elif choice == "3":
        csv_file = input("Pfad zur CSV-Datei: ").strip()
        if not os.path.exists(csv_file):
            print("Datei nicht gefunden!")
            return
        
        # Teste Parser mit einzelner Datei
        with open(csv_file, 'r', encoding='utf-8') as f:
            csv_content = f.read()
        
        analysis = analyzer.parse_bundesbank_csv(csv_content, "TEST")
        
        print("\n=== PARSING-ERGEBNIS ===")
        print(f"Status: {analysis.get('parsing_status', 'unknown')}")
        
        if analysis.get('parsing_status') == 'success':
            print("\nExtrahierte Metadaten:")
            metadata_fields = ['dezimalstellen', 'dimension', 'einheit', 'format_zeitangabe', 'kategorie', 'stand_vom']
            for field in metadata_fields:
                if field in analysis:
                    print(f"- {field}: {analysis[field]}")
            
            print("\nZeitreihen-Info:")
            time_fields = ['min_datum', 'max_datum', 'total_periods', 'estimated_frequency', 'data_completeness_percent']
            for field in time_fields:
                if field in analysis:
                    print(f"- {field}: {analysis[field]}")
            
            if 'latest_value' in analysis:
                print(f"\nAktuelle Daten:")
                print(f"- Letzter Zeitpunkt: {analysis.get('latest_period', 'N/A')}")
                print(f"- Letzter Wert: {analysis.get('latest_value', 'N/A')}")
                if 'latest_change_percent' in analysis:
                    print(f"- Letzte Veränderung: {analysis['latest_change_percent']:.2f}%")
        else:
            print(f"Parsing fehlgeschlagen: {analysis.get('parsing_error', 'Unbekannter Fehler')}")
    
    else:
        print("Ungültige Auswahl!")


if __name__ == "__main__":
    main()

=== BUNDESBANK DATENANALYSE TOOL (Erweitert) ===
Extrahiert automatisch:
- Dezimalstellen, Dimension, Einheit
- Format der Zeitangabe, Kategorie, Stand vom
- Min-Datum, Max-Datum (automatisch ermittelt)
- Vollständige Zeitreihen-Analyse

1. Erweitere Scraping-Ergebnisse (Download + Bundesbank-Analyse)
2. Generiere Bundesbank-Datenreport
3. Teste einzelne CSV-Datei


c:\Users\marius.brede\.conda\envs\general\Lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(



Starte Bundesbank-Analyse...
Extrahiert: Dezimalstellen, Dimension, Einheit, Zeitangaben, etc.


2025-08-13 11:23:46,293 - INFO - Scraping-Datei geladen: 34978 Einträge
2025-08-13 11:23:46,326 - INFO - Verarbeite 34874 erfolgreiche Einträge
2025-08-13 11:23:46,334 - INFO - Verarbeite: BBBK1.M.OUA249
2025-08-13 11:23:48,417 - INFO - Verarbeite: BBBK1.M.OUA250
2025-08-13 11:23:49,679 - INFO - Verarbeite: BBBK1.M.OUA251
2025-08-13 11:23:50,897 - INFO - Verarbeite: BBBK1.M.OUA252
2025-08-13 11:23:52,140 - INFO - Verarbeite: BBBK1.M.OUA253
2025-08-13 11:23:53,339 - INFO - Verarbeite: BBBK1.M.OUA284
2025-08-13 11:23:54,481 - INFO - Verarbeite: BBBK1.M.OUA285
2025-08-13 11:23:55,722 - INFO - Verarbeite: BBBK1.M.OUA295
2025-08-13 11:23:56,904 - INFO - Verarbeite: BBBK1.M.OUA298
2025-08-13 11:23:58,054 - INFO - Verarbeite: BBBK1.M.OUA300
2025-08-13 11:23:59,259 - INFO - Verarbeite: BBBK1.M.OUA301
2025-08-13 11:23:59,259 - INFO - Fortschritt: 10/34874 abgeschlossen
2025-08-13 11:24:00,380 - INFO - Verarbeite: BBBK1.M.OUA302
2025-08-13 11:24:01,614 - INFO - Verarbeite: BBBK1.M.OUA303
2025-08


=== BUNDESBANK-ANALYSE ABGESCHLOSSEN ===
Ausgabedatei: enhanced_data\enhanced_bundesbank_data_20250813_222018.xlsx

Bundesbank-Metadaten extrahiert:
- Erfolgreiches Parsing: 34874
- Mit Dimension/Einheit: 34874
- Mit Zeitreihen-Daten: 34874

Beispiel-Metadaten:
- Dimension: Milliarden
- Einheit: DM/Euro
- Dezimalstellen: 3
- Zeitspanne: 1968-12 bis 2025-06
- Frequenz: Monthly


2025-08-13 22:27:48,152 - INFO - Bundesbank-Datenreport erstellt: enhanced_data\enhanced_bundesbank_data_20250813_222018_report.json



Bundesbank-Report erstellt: enhanced_data\enhanced_bundesbank_data_20250813_222018_report.json
- Metadaten-Extraktionsrate: 100.0%
- Unique Dimensionen: 4
- Unique Einheiten: 16
